In [0]:
# ============================================
# STEP 1 — Install Libraries
# ============================================
%pip install openmeteo-requests requests-cache retry-requests

In [0]:
# ============================================
# STEP 2 — Imports
# ============================================
import openmeteo_requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

def retry(session, retries, backoff_factor):
    retry_strategy = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

In [0]:
# ============================================
# STEP 3 — Fetch 5 Years Weather Data (Luxembourg)
# ============================================
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta

cache = requests_cache.CachedSession(".cache", expire_after=-1)
om = openmeteo_requests.Client(session=retry(cache, retries=5, backoff_factor=0.2))

END   = datetime.today().strftime("%Y-%m-%d")
START = (datetime.today() - timedelta(days=365*5)).strftime("%Y-%m-%d")

params = {
    "latitude":   49.6117,
    "longitude":  6.1319,
    "start_date": START,
    "end_date":   END,
    "hourly": [
        "temperature_2m",           # 0
        "apparent_temperature",     # 1
        "wind_speed_10m",           # 2
        "wind_speed_100m",          # 3
        "wind_direction_10m",       # 4
        "wind_gusts_10m",           # 5
        "shortwave_radiation",      # 6
        "direct_radiation",         # 7
        "diffuse_radiation",        # 8
        "precipitation",            # 9
        "rain",                     # 10
        "snowfall",                 # 11
        "snow_depth",               # 12
        "cloud_cover",              # 13
        "relative_humidity_2m",     # 14
        "dew_point_2m",             # 15
        "pressure_msl",             # 16
        "sunshine_duration",        # 17
        "soil_temperature_0_to_7cm" # 18
    ],
    "timezone": "Europe/Luxembourg"
}

r = om.weather_api(
    "https://archive-api.open-meteo.com/v1/archive",
    params=params
)[0]

h = r.Hourly()

# Use periods= instead of end= to guarantee length matches data arrays
timestamps = pd.date_range(
    start=pd.to_datetime(h.Time(), unit="s", utc=True),
    periods=len(h.Variables(0).ValuesAsNumpy()),
    freq=pd.Timedelta(seconds=h.Interval())
)

df = pd.DataFrame({
    "timestamp":           timestamps,
    "temperature_2m":      h.Variables(0).ValuesAsNumpy(),
    "apparent_temp":       h.Variables(1).ValuesAsNumpy(),
    "wind_speed_10m":      h.Variables(2).ValuesAsNumpy(),
    "wind_speed_100m":     h.Variables(3).ValuesAsNumpy(),
    "wind_direction_10m":  h.Variables(4).ValuesAsNumpy(),
    "wind_gusts_10m":      h.Variables(5).ValuesAsNumpy(),
    "shortwave_radiation": h.Variables(6).ValuesAsNumpy(),
    "direct_radiation":    h.Variables(7).ValuesAsNumpy(),
    "diffuse_radiation":   h.Variables(8).ValuesAsNumpy(),
    "precipitation":       h.Variables(9).ValuesAsNumpy(),
    "rain":                h.Variables(10).ValuesAsNumpy(),
    "snowfall":            h.Variables(11).ValuesAsNumpy(),
    "snow_depth":          h.Variables(12).ValuesAsNumpy(),
    "cloud_cover":         h.Variables(13).ValuesAsNumpy(),
    "relative_humidity":   h.Variables(14).ValuesAsNumpy(),
    "dew_point":           h.Variables(15).ValuesAsNumpy(),
    "pressure_msl":        h.Variables(16).ValuesAsNumpy(),
    "sunshine_duration":   h.Variables(17).ValuesAsNumpy(),
    "soil_temp_0_7cm":     h.Variables(18).ValuesAsNumpy(),
})

# Convert timezone from UTC to local Luxembourg time
df["timestamp"] = df["timestamp"].dt.tz_convert("Europe/Luxembourg")

print(f"Downloaded {len(df):,} hourly rows | {len(df.columns)} variables | {START} to {END}")
df.head(10)

# 📘 Data Dictionary — SudEnergy Luxembourg Weather Dataset

This dataset contains 5 years of hourly weather data for **Luxembourg City** (49.61°N, 6.13°E)  
sourced from [Open-Meteo Historical Weather API](https://open-meteo.com).  
Total records: **43,824 hourly rows** | Period: **2021–2026**

---

## 🕐 Time Columns
| Column | Unit | Description |
|--------|------|-------------|
| `date` | YYYY-MM-DD | Date of observation |
| `time` | HH:MM:SS | Time of observation |
| `hour` | 0–23 | Hour as integer, useful for modeling |

## 🌡️ Temperature
| Column | Unit | Description |
|--------|------|-------------|
| `temperature_2m` | °C | Actual air temperature at 2m height |
| `apparent_temp` | °C | Feels-like temperature (wind chill + humidity) |

## 💨 Wind
| Column | Unit | Description |
|--------|------|-------------|
| `wind_speed_10m` | km/h | Wind speed at 10m — standard meteorological level |
| `wind_speed_100m` | km/h | Wind speed at 100m — relevant for wind turbines |
| `wind_direction_10m` | degrees | 0°=North, 90°=East, 180°=South, 270°=West |
| `wind_gusts_10m` | km/h | Peak wind gusts — important for grid stability |

## ☀️ Solar
| Column | Unit | Description |
|--------|------|-------------|
| `shortwave_radiation` | W/m² | Total solar energy reaching the surface |
| `direct_radiation` | W/m² | Direct sunlight — key for solar panel estimation |
| `diffuse_radiation` | W/m² | Scattered/cloudy light |
| `sunshine_duration` | seconds | Actual sunshine seconds per hour |

## 🌧️ Precipitation
| Column | Unit | Description |
|--------|------|-------------|
| `precipitation` | mm | Total precipitation (rain + snow) |
| `rain` | mm | Liquid rain only |
| `snowfall` | cm | Snowfall amount |
| `snow_depth` | cm | Snow on ground — drives heating demand spikes |

## 🌫️ Atmosphere
| Column | Unit | Description |
|--------|------|-------------|
| `cloud_cover` | % | Percentage of sky covered by clouds |
| `relative_humidity` | % | Humidity — affects comfort and cooling demand |
| `dew_point` | °C | Temperature at which condensation forms |
| `pressure_msl` | hPa | Atmospheric pressure — indicates incoming weather systems |

## 🌱 Soil
| Column | Unit | Description |
|--------|------|-------------|
| `soil_temp_0_7cm` | °C | Ground temperature at 0–7cm — relevant for building heating/cooling |


In [0]:
# ============================================
# STEP 4 — Save Raw Data (Bronze Layer)
# ============================================

# Strip timezone info before passing to Spark
df_spark = df.copy()
df_spark["timestamp"] = df_spark["timestamp"].dt.tz_localize(None)

spark_df = spark.createDataFrame(df_spark)
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.bronze_weather_hourly")

print(f"Saved {len(df):,} rows to workspace.default.bronze_weather_hourly")

In [0]:
# ============================================
# STEP 4b — Preview Bronze Data
# ============================================
display(spark.sql("SELECT * FROM workspace.default.bronze_weather_hourly LIMIT 10"))

In [0]:
# ============================================
# STEP 4c — Separate Date and Time Columns
# ============================================
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["date"] = df["timestamp"].dt.strftime("%Y-%m-%d")
df["time"] = df["timestamp"].dt.strftime("%H:%M:%S")

# Drop timestamp
df2 = df.drop(columns=["timestamp"])

# Reorder
cols = ["date", "time"] + [c for c in df2.columns if c not in ["date","time"]]
df2 = df2[cols]

# Check
print(df2.head(3))

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_weather_hourly")

spark_df = spark.createDataFrame(df2)
spark_df.write.mode("overwrite").saveAsTable("workspace.default.bronze_weather_hourly")

spark.table("workspace.default.bronze_weather_hourly").printSchema()

In [0]:
display(spark.table("workspace.default.bronze_weather_hourly").orderBy("date", "time"))

In [0]:
# ============================================
# STEP 5 — Create Silver Layer (Daily Aggregation)
# ============================================
df_bronze = spark.table("workspace.default.bronze_weather_hourly").toPandas()
df_bronze["date"] = pd.to_datetime(df_bronze["date"]).dt.normalize()

daily = df_bronze.groupby("date").agg(
    avg_temp=("temperature_2m", "mean"),
    max_temp=("temperature_2m", "max"),
    min_temp=("temperature_2m", "min"),
    avg_apparent_temp=("apparent_temp", "mean"),
    avg_wind_10m=("wind_speed_10m", "mean"),
    max_wind_10m=("wind_speed_10m", "max"),
    avg_wind_100m=("wind_speed_100m", "mean"),
    max_wind_gusts=("wind_gusts_10m", "max"),
    avg_wind_direction=("wind_direction_10m", "mean"),
    total_radiation=("shortwave_radiation", "sum"),
    total_direct_radiation=("direct_radiation", "sum"),
    total_diffuse_radiation=("diffuse_radiation", "sum"),
    total_sunshine=("sunshine_duration", "sum"),
    total_precipitation=("precipitation", "sum"),
    total_rain=("rain", "sum"),
    total_snowfall=("snowfall", "sum"),
    max_snow_depth=("snow_depth", "max"),
    avg_cloud_cover=("cloud_cover", "mean"),
    avg_humidity=("relative_humidity", "mean"),
    avg_dew_point=("dew_point", "mean"),
    avg_pressure=("pressure_msl", "mean"),
    avg_soil_temp=("soil_temp_0_7cm", "mean"),
).reset_index()

# Energy-relevant features
BASE = 15.5
daily["HDD"]        = (BASE - daily["avg_temp"]).clip(lower=0)
daily["CDD"]        = (daily["avg_temp"] - BASE).clip(lower=0)
daily["season"]     = daily["date"].dt.month.map({
    12:"Winter", 1:"Winter",  2:"Winter",
    3:"Spring",  4:"Spring",  5:"Spring",
    6:"Summer",  7:"Summer",  8:"Summer",
    9:"Autumn",  10:"Autumn", 11:"Autumn"
})
daily["is_weekend"] = daily["date"].dt.dayofweek.isin([5,6]).astype(int)
daily["month"]      = daily["date"].dt.month
daily["year"]       = daily["date"].dt.year
daily["dayofweek"]  = daily["date"].dt.dayofweek

# Convert date to string to avoid timestamp format in Spark
daily["date"] = daily["date"].dt.strftime("%Y-%m-%d")
# Save
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_weather_daily")
spark_df = spark.createDataFrame(daily)
spark_df.write.mode("overwrite").saveAsTable("workspace.default.silver_weather_daily")

print(f"Silver table saved: {len(daily)} daily rows")
print(f"Columns: {daily.columns.tolist()}")
display(spark.table("workspace.default.silver_weather_daily"))

## 📊 Data Layers Explanation

### 🟤 Bronze Layer — Hourly Raw Data
- `date` + `time` columns — one row per hour
- Raw values — exact sensor readings e.g. `16.7°C` at `10:00:00`
- **43,824 rows total**

### 🥈 Silver Layer — Daily Aggregated Data
- `date` only — one row per day
- Values are aggregated:
  - `avg_temp` — average of all 24 hourly readings for that day
  - `max_temp` — highest temperature reading of the day
  - `min_temp` — lowest temperature reading of the day
  - `avg_wind_10m` — average wind speed across the day
  - `max_wind_10m` — peak wind speed of the day
  - `HDD / CDD` — Heating/Cooling Degree Days, energy demand indicators
  - `season` — Winter / Spring / Summer / Autumn
- **1,826 rows total**

### Why aggregation matters for forecasting?
Prophet requires **one value per day** to learn seasonal patterns.  
Aggregating from hourly → daily removes noise and gives the model clean, meaningful signals.

In [0]:
# ============================================
# STEP 6 — Data Quality Check (Silver Layer)
# ============================================
import pandas as pd
import numpy as np

daily = spark.table("workspace.default.silver_weather_daily").toPandas()
daily["date"] = pd.to_datetime(daily["date"])

print("=" * 50)
print("1. SHAPE & COVERAGE")
print("=" * 50)
print(f"Rows:          {len(daily)}")
print(f"Columns:       {len(daily.columns)}")
print(f"Date range:    {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"Days expected: {(daily['date'].max() - daily['date'].min()).days + 1}")
print(f"Days actual:   {len(daily)}")
print(f"Missing days:  {(daily['date'].max() - daily['date'].min()).days + 1 - len(daily)}")

print("\n" + "=" * 50)
print("2. MISSING VALUES")
print("=" * 50)
missing = daily.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✓")

print("\n" + "=" * 50)
print("3. DUPLICATES")
print("=" * 50)
dups = daily["date"].duplicated().sum()
print(f"Duplicate dates: {dups}" if dups > 0 else "No duplicates ✓")

print("\n" + "=" * 50)
print("4. OUTLIERS (beyond 3 std)")
print("=" * 50)
num_cols = daily.select_dtypes(include=np.number).columns
for col in num_cols:
    mean = daily[col].mean()
    std  = daily[col].std()
    outliers = daily[(daily[col] > mean + 3*std) | (daily[col] < mean - 3*std)]
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers | min={daily[col].min():.2f} max={daily[col].max():.2f}")

print("\n" + "=" * 50)
print("5. DATE CONTINUITY")
print("=" * 50)
daily_sorted = daily.sort_values("date")
date_diff = daily_sorted["date"].diff().dt.days
gaps = date_diff[date_diff > 1]
print("No gaps in date sequence ✓" if len(gaps) == 0 else f"Gaps found:\n{gaps}")

print("\n" + "=" * 50)
print("6. VALUE RANGES")
print("=" * 50)
print(daily.describe().round(2))

**✅ All good:**

1,827 rows — perfect, no missing days
No missing values
No duplicates
Date continuity perfect

**⚠️ Outliers — but all are REAL, do not remove them:**

- max_wind_gusts: 92 km/h — real storm, Luxembourg gets these
- total_precipitation: 70.6mm — real heavy rain event
- total_snowfall: 10cm — real snowfall
- avg_pressure: 982 hPa — real low pressure storm system
- CDD: 12.07 — real hot summer day

**These are not errors — they are exactly the extreme weather events that drive energy demand spikes.**

In [0]:
print(daily[["max_snow_depth"]].describe())
print(daily[daily["max_snow_depth"] > 0]["max_snow_depth"].sort_values(ascending=False).head(10))

In [0]:
# ============================================
# STEP 6 — Data Quality Check
# ============================================
daily = spark.table("workspace.default.bronze_weather_hourly").toPandas()

print("=== SHAPE ===")
print(daily.shape)

print("\n=== DATA TYPES ===")
print(daily.dtypes)

print("\n=== MISSING VALUES ===")
print(daily.isnull().sum())

print("\n=== ZERO VALUES ===")
print((daily == 0).sum())

print("\n=== BASIC STATISTICS ===")
print(daily.describe())
  

In [0]:
# ============================================
# Exploratory Analysis & Charts
# ============================================

# Reload from saved table and perform aggregations
df_hourly = spark.table("workspace.default.bronze_weather_hourly").toPandas()


# Aggregate to daily
daily = df_hourly.groupby("date").agg({
    "temperature_2m": "mean",
    "shortwave_radiation": "sum",
    "precipitation": "sum"
}).reset_index()

# Rename columns
daily = daily.rename(columns={
    "temperature_2m": "avg_temp",
    "shortwave_radiation": "total_radiation"
})

# Add date components
daily["month"] = daily["date"].dt.month
daily["year"] = daily["date"].dt.year

# Calculate degree days (base 18°C)
daily["HDD"] = np.maximum(18 - daily["avg_temp"], 0)
daily["CDD"] = np.maximum(daily["avg_temp"] - 18, 0)

# Calculate rolling temperature
daily["rolling_temp"] = daily["avg_temp"].rolling(30, center=True).mean()

# Plot 1: Rolling temperature trend
fig1 = plt.figure(figsize=(12, 5))
plt.plot(daily["date"], daily["rolling_temp"], color="#185FA5", lw=1.5)
plt.title("30-day Rolling Temperature (°C)", fontsize=12, fontweight="bold")
plt.ylabel("°C")
plt.xlabel("Date")
plt.grid(alpha=0.3)
plt.tight_layout()
display(fig1)
plt.close()

# Plot 2: Monthly HDD boxplot
fig2 = plt.figure(figsize=(12, 5))
daily.boxplot(column="HDD", by="month", ax=plt.gca())
plt.title("Heating Degree Days by Month", fontsize=12, fontweight="bold")
plt.suptitle("")  # Remove default title
plt.xlabel("Month")
plt.ylabel("HDD")
plt.tight_layout()
display(fig2)
plt.close()

# Plot 3: Annual HDD vs CDD
fig3 = plt.figure(figsize=(12, 5))
ann = daily.groupby("year")[["HDD","CDD"]].sum()
ann.plot(kind="bar", ax=plt.gca(), color=["#185FA5","#D85A30"], rot=0)
plt.title("Annual HDD vs CDD", fontsize=12, fontweight="bold")
plt.ylabel("Degree days")
plt.xlabel("Year")
plt.legend(["HDD", "CDD"])
plt.tight_layout()
display(fig3)
plt.close()

# Plot 4: Solar radiation by month
fig4 = plt.figure(figsize=(12, 5))
monthly_rad = daily.groupby("month")["total_radiation"].mean()
plt.bar(monthly_rad.index, monthly_rad.values, color="#EF9F27")
plt.title("Average Solar Radiation by Month (W/m²)", fontsize=12, fontweight="bold")
plt.ylabel("W/m²")
plt.xlabel("Month")
plt.xticks(range(1,13), ["Jan","Feb","Mar","Apr","May","Jun",
                         "Jul","Aug","Sep","Oct","Nov","Dec"])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig4)
plt.close()